# AML Assignment 1 - Bird classification
ResNet-101 vs EfficientNet-B2 vs Swin-Tiny

Run the setup cells top to bottom at the start of every session.

## 1. Check the GPU
Runtime -> Change runtime type -> **T4 GPU**

In [ ]:
!nvidia-smi
import os
print('CPU cores:', os.cpu_count())

## 2. Mount Drive and clone the repo
Add a Colab secret named `GITHUB_TOKEN` (key icon in the sidebar) if the repo is private.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

GITHUB_USER = 'jysharon-lee'
REPO = 'COS30082---Applied-Machine-Learning'

import os
if not os.path.exists('/content/repo'):
    try:
        token = userdata.get('GITHUB_TOKEN')       # needed only if the repo is private
        url = f'https://{token}@github.com/{GITHUB_USER}/{REPO}.git'
    except Exception:
        url = f'https://github.com/{GITHUB_USER}/{REPO}.git'   # public repo
    !git clone {url} /content/repo
else:
    !cd /content/repo && git pull


## 3. Locate the dataset
The images and txt files come with the cloned repo (`AML Assignment 1/Train`, `Test`, `train.txt`, `test.txt`), so no unzip or Drive copy is needed.

In [ ]:
!ls "/content/repo/AML Assignment 1"
!echo "Train images:" && find "/content/repo/AML Assignment 1/Train" -type f | wc -l
!echo "Test images:"  && find "/content/repo/AML Assignment 1/Test"  -type f | wc -l


## 4. Install requirements and make `src` importable

In [ ]:
%cd /content/repo
!pip install -q -r requirements.txt

import sys
sys.path.insert(0, '/content/repo')

%load_ext autoreload
%autoreload 2

## 5. Point the config at the Colab paths
`TRAIN_TXT` and `TEST_TXT` follow `DATA_ROOT` automatically. Checkpoints and results go to Drive so they survive a disconnect.

In [ ]:
from pathlib import Path
from src.config import CFG, device, seed_everything

CFG.DATA_ROOT   = Path('/content/repo/AML Assignment 1')
CFG.CKPT_DIR    = Path('/content/drive/MyDrive/AML/checkpoints')
CFG.RESULTS_DIR = Path('/content/drive/MyDrive/AML/results')
CFG.CKPT_DIR.mkdir(parents=True, exist_ok=True)
CFG.RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CFG.BATCH_SIZE = 32        # T4 16GB; try 64 for ResNet-101 / EfficientNet-B2
seed_everything()
print(device, '| train.txt found:', CFG.TRAIN_TXT.exists(), '| test.txt found:', CFG.TEST_TXT.exists())

## 6. Sanity check: models and data

In [ ]:
import torch
from src.models import build_model, count_params
from src.data import make_loaders, MODEL_RES

for name in ['resnet101', 'efficientnet_b2', 'swin_t']:
    model = build_model(name)
    res = 224 if CFG.UNIFORM_RES else MODEL_RES[name]
    x = torch.randn(4, 3, res, res, device=device).contiguous(memory_format=torch.channels_last)
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
        out = model(x)
    tot, trn = count_params(model)
    print(f'{name:16s} out={tuple(out.shape)} params={tot:.1f}M trainable={trn:.1f}M')
    del model; torch.cuda.empty_cache()

In [ ]:
train_loader, val_loader, test_loader = make_loaders('resnet101')
xb, yb = next(iter(train_loader))
print('batch:', xb.shape, '| label range:', yb.min().item(), '-', yb.max().item())

## 7. Training (next step)
`src/train.py` with per-epoch checkpointing to Drive goes here.